Data source : https://www.kaggle.com/c/home-credit-default-risk/data

But notebook : Feature Engineering / Reduction

- utiliser les donnes agrégées précédemment
- réduire le dataframe (optimisation de futurs temps de calcul / temps de modélisation) tout en ayant le moins de données manquantes dans les features restantes
   - verification des types et choix de types plus avantageux en poids si possible
   - entrainement sur différents modèles et extraction des features notables via SHAPE ou autre (knn most important features)
   - éviter de "créer" des informations ex-nihilo avec de l'inputing : favoriser la sélection / suppression en préservant au mieux l'intégrité des données

# Imports

## libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import numpy as np
import pandas as pd
import inspect

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# import utils from parent directory
import sys
from pathlib import Path
parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

import utils.UVA as UVA
import utils.MVA as MVA
import utils.PCA as PCA

In [ ]:
import gc
import time
from contextlib import contextmanager
import re

In [ ]:
# https://numpy.org/doc/stable/release/2.0.0-notes.html#representation-of-numpy-scalars-changed
np.set_printoptions(legacy='1.21')

In [ ]:
import mlflow
import mlflow.lightgbm
import mlflow.xgboost
import mlflow.sklearn

from mlflow import MlflowClient
from mlflow.models.signature import infer_signature
from pprint import pprint

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer

import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn import model_selection
import xgboost as xgb


from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, fbeta_score, make_scorer

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


import shap

## data

In [ ]:
csv_path_output = '../data/output/'
file_name = 'df_cleaned.csv'

df = pd.read_csv(f'{csv_path_output + file_name}', encoding='ISO-8859-1')
df.drop(df.filter(regex="Unname"),axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
df.head()

# Explo

## Valeurs manquantes

In [ ]:
# https://medium.com/@kavyasree42/how-to-handle-missing-data-in-python-simple-to-advanced-techniques-machine-learning-brain-ffae3c1a694f

In [ ]:
@contextmanager
def timer(title):
    t0 = time.time()
    yield
    print("{} - done in {:.0f}s".format(title, time.time() - t0))
    
# Function to calculate missing values by column
def missing_values_table(df):
    # Taux de valeurs manquantes :
    print("Taux de valeurs manquantes dans le df avant imputation :" , round(df.isna().mean().mean(),4))
    # Total missing values
    mis_val = df.isnull().sum()
    
    # Percentage of missing values
    mis_val_percent = 100 * df.isnull().sum() / len(df)
    
    # Make a table with the results
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
    
    # Rename the columns
    mis_val_table_ren_columns = mis_val_table.rename(columns = {0 : 'Missing Values', 1 : '% of Total Values'})
    
    # Sort the table by percentage of missing descending
    mis_val_table_ren_columns = mis_val_table_ren_columns[
        mis_val_table_ren_columns.iloc[:,1] != 0].sort_values(
        '% of Total Values', ascending=False).round(1)
    
    # Print some summary information
    print ("Your selected dataframe has " + str(df.shape[1]) + " columns.\n"
           "There are " + str(mis_val_table_ren_columns.shape[0]) +
           " columns that have missing values.")
    
    # Return the dataframe with missing information
    return mis_val_table_ren_columns

In [ ]:
#  Statistiques sur les valeurs manquantes :
missing_values = missing_values_table(df)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Séparation du jeu de données en jeu de  training/validation et jeu de test
train_df = df[df['TARGET'].notnull()] # partie du jeu de données qui contient une target
test_df = df[df['TARGET'].isnull()] # partie du jeu de données qui n'a pas de target
print(train_df.shape, test_df.shape)
train_df["TARGET"].value_counts()/train_df.shape[0]

# Taux de valeurs manquantes :
print("Taux de valeurs manquantes dans jeu de train avant imputation :" , round(train_df.isna().mean().mean(),4))
print("Taux de valeurs manquantes dans jeu de test avant imputation :" , round(test_df.isna().mean().mean(),4))

# Typage

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
# Vérification des dtypes actuels
print("Dtypes initiaux :")
print(df.dtypes.value_counts())
print("\nInfos avant conversion :")
print(df.info())
df_memory_usage_gb = df.memory_usage(deep=True).sum() / (1024 ** 3)
print(f'{df_memory_usage_gb} Gb')

# Analyse de la distribution des données
for column in df.columns:
    if df[column].dtype == 'int64':
        min_val, max_val = df[column].min(), df[column].max()
        if min_val >= np.iinfo(np.int8).min and max_val <= np.iinfo(np.int8).max:
            df[column] = df[column].astype(np.int8)
        elif min_val >= np.iinfo(np.int16).min and max_val <= np.iinfo(np.int16).max:
            df[column] = df[column].astype(np.int16)
        elif min_val >= np.iinfo(np.int32).min and max_val <= np.iinfo(np.int32).max:
            df[column] = df[column].astype(np.int32)
    elif df[column].dtype == 'float64':
        df[column] = df[column].astype(np.float32)

# Vérification des dtypes après conversion
print("\nDtypes après conversion :")
print(df.dtypes.value_counts())
print("\nInfos sur le DataFrame :")
print(df.info())
df_memory_usage_gb = df.memory_usage(deep=True).sum() / (1024 ** 3)
print(f'{df_memory_usage_gb} Gb')

In [ ]:
df['TARGET'].value_counts(dropna=False)

In [ ]:
fig,ax=plt.subplots(figsize=(7,6))
plt.title("Distribution de la cible)", fontsize=15)
sns.countplot(x='TARGET', data=df)
for container in ax.containers:
    ax.bar_label(container,size=12,fmt='%.0f')
plt.show()

In [ ]:
# Séparer les données en X_train et X_test
X_train = df[df['TARGET'].notna()]
X_test = df[df['TARGET'].isna()]

print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test :", X_test.shape)

In [ ]:
X_train['TARGET'].value_counts(dropna=False)

In [ ]:
# Vérifier les valeurs manquantes dans X_train
missing_values = X_train.isna().sum()

# Afficher les valeurs manquantes par colonne
print("Valeurs manquantes dans X_train :")
print(missing_values.sum())

In [ ]:
# Calculer le pourcentage de valeurs manquantes par colonne
missing_percentages = X_train.isna().mean() * 100

# Créer un DataFrame pour afficher les résultats
missing_table = pd.DataFrame({
    'Feature': missing_percentages.index,
    'Missing Percentage': missing_percentages.values
}).sort_values(by='Missing Percentage', ascending=False)

# Ajouter une ligne pour le total
total_missing_percentage = missing_percentages.mean()
missing_table = pd.concat([missing_table, pd.DataFrame({
    'Feature': ['Total'],
    'Missing Percentage': [total_missing_percentage]
})], ignore_index=True)

# Afficher le tableau
pd.set_option('display.max_rows', None)
print("Pourcentage de valeurs manquantes par feature :")
print(missing_table)

In [ ]:
def missing_values_count(df):
    # Calculer le pourcentage de valeurs manquantes par colonne
    missing_percentages = df.isna().mean() * 100
    
    # Créer des bins pour les pourcentages de valeurs manquantes
    bins = [0, 0.01, 20, 40, 60, 100]  # 0%, 0-20%, 20-40%, 40-60%, 60-100%
    labels = ['0%', '0-20%', '20-40%', '40-60%', '60-100%']
    
    # Classer les caractéristiques dans les bins
    missing_bins = pd.cut(missing_percentages, bins=bins, labels=labels, right=False)
    
    # Compter le nombre de variables par bin
    bin_counts = missing_bins.value_counts().sort_index()
    
    # Calculer le pourcentage de variables dans chaque bin par rapport au total
    total_variables = len(missing_percentages)
    bin_percentages = (bin_counts / total_variables) * 100
    
    # Créer un DataFrame pour afficher les résultats
    bin_summary = pd.DataFrame({
        'Number of Variables': bin_counts,
        'Percentage of Total': bin_percentages
    })
    
    # Afficher le nombre de variables par bin et leur pourcentage par rapport au total
    print("Nombre de variables par bin de valeurs manquantes et leur pourcentage par rapport au total :")
    print(bin_summary)

In [ ]:
missing_values_count(X_train)

# Modélisations + MLflow

In [ ]:
# Séparer les caractéristiques et la cible
X = X_train.drop(columns=['TARGET'])
y = X_train['TARGET']

# Remplacer les valeurs infinies par NaN ___ "XGBoostError : Check failed: valid: Input data contains `inf` or a value too large, while `missing` is not set to `inf`"
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remplacer les NaN par la moyenne de la colonne
#X.fillna(X.mean(), inplace=True)

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# Diviser les données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## test MLflow

In [ ]:
# Définir les paramètres pour chaque modèle
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'max_depth': 3,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'max_depth': 3,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

rf_params = {
    'n_estimators': 100,
    'max_depth': 3,
    'random_state': 42
}

In [ ]:
# Fonction pour entraîner et suivre un modèle avec MLflow
def train_and_log_model(model_name, model, params):
    with mlflow.start_run():
        # Entraîner le modèle
        model.fit(X_train, y_train)

        # Prédire les probabilités pour la classe positive
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        # Calculer le score ROC-AUC
        auc_roc = roc_auc_score(y_test, y_pred_proba)

        # Journaliser les paramètres et les métriques
        mlflow.log_params(params)
        mlflow.log_metric("auc_roc", auc_roc)

        # Prendre un exemple d'entrée pour la signature du modèle
        input_example = X_train.head(1)

        # Journaliser le modèle avec un exemple d'entrée
        if model_name == "XGBoost":
            mlflow.xgboost.log_model(model, "model", input_example=input_example)
        elif model_name == "LightGBM":
            mlflow.lightgbm.log_model(model, "model", input_example=input_example)
        elif model_name == "RandomForest":
            mlflow.sklearn.log_model(model, "model", input_example=input_example)

        print(f"{model_name} - AUC-ROC: {auc_roc}")


In [ ]:
# Entraîner et suivre chaque modèle
train_and_log_model("XGBoost", xgb.XGBClassifier(**xgb_params), xgb_params)

In [ ]:
train_and_log_model("LightGBM", lgb.LGBMClassifier(**lgb_params), lgb_params)

In [ ]:
train_and_log_model("RandomForest", RandomForestClassifier(**rf_params), rf_params)

In [ ]:
# Initialiser MLflow
mlflow.set_experiment("Model_Comparison")

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

def train_model(model, param_dist, X_train, y_train, X_test, y_test, model_name):
    with mlflow.start_run():
        # RandomizedSearchCV pour le tuning des hyperparamètres
        random_search = RandomizedSearchCV(
            model, param_distributions=param_dist, n_iter=10, cv=5, scoring='accuracy', random_state=42
        )
        random_search.fit(X_train, y_train)

        # Enregistrer les paramètres et les métriques avec MLflow
        mlflow.log_params(random_search.best_params_)
        metrics = evaluate_model(random_search.best_estimator_, X_test, y_test)
        mlflow.log_metrics(metrics)

        # Enregistrer le modèle avec un exemple d'entrée
        input_example = X_train[:1]  # Utiliser la première ligne de X_train comme exemple d'entrée
        mlflow.sklearn.log_model(
            random_search.best_estimator_, model_name, input_example=input_example
        )

        return random_search.best_estimator_, metrics


In [ ]:
# Définir les modèles et les espaces de recherche des hyperparamètres
models = {
    "XGBoost": {
        "model": XGBClassifier(),
        "param_dist": {
            "n_estimators": [100, 200, 300],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.1, 0.2],
        }
    },
    "LightGBM": {
        "model": LGBMClassifier(),
        "param_dist": {
            "n_estimators": [100, 200, 300],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.1, 0.2],
            "num_leaves": [20, 30, 40, 50], 
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(),
        "param_dist": {
            "n_estimators": [100, 200, 300],
            "max_depth": [3, 5, 7],
            "min_samples_split": [2, 5, 10],
        }
    }
}

In [ ]:
# Entraîner et évaluer les modèles
for model_name, model_info in models.items():
    print(f"Training {model_name}...")
    best_model, metrics = train_model(
        model_info["model"], model_info["param_dist"], X_train, y_train, X_test, y_test, model_name
    )
    print(f"{model_name} metrics: {metrics}")